# Thalamus — fish species classifier (Colab)

Colab wrapper around `scripts/fish_classifier_train.py`. The training logic lives
in the repo, not in this notebook, so a run here matches a run on a workstation.

**Runtime → Change runtime type → GPU** before you start.

## How this notebook flows

Run the sections in order, top to bottom. There is exactly one decision point:

```
1. Check the GPU
2. Clone the repo + install dependencies      <- must run before anything else
3. Get raw images        CHOOSE ONE:  3A build from iNaturalist
                                      3B load an archive from Drive
4. Prepare images        audit, then standardise
5. Split into train/val/test
6. Verify the layout     <- stop here if this fails; training cannot work
7. Configure
8. Train
9. Back up checkpoints
10. Export to TFLite
11. Download app artifacts
```

Steps 3A and 3B are alternatives — run one, not both. Everything else runs once,
in sequence.

The data ends up in two places, which is deliberate:

- `/content/raw_fish/<species>/*.jpg` — raw downloads, one folder per species
- `/content/Thalamus/data/{train,val,test}/<species>/*.jpg` — what training reads

Step 5 turns the first into the second. Training reads only the second.

## 1. Check the GPU

If this reports no GPU, training still runs but roughly an order of magnitude
slower.

In [ ]:
!nvidia-smi || echo 'No GPU allocated — check Runtime > Change runtime type'

## 2. Clone the repo and install dependencies

**Run this before any other section.** Later cells call scripts from the repo, so
nothing works until it exists.

Thalamus is a **private** repository, so Colab cannot clone it anonymously — an
anonymous clone fails with `could not read Username for 'https://github.com'`.
Give Colab a token once and it is remembered across notebooks:

1. Create a GitHub personal access token at
   <https://github.com/settings/tokens>. A fine-grained token needs
   **Contents: Read-only** on this repository; a classic token needs the
   **repo** scope.
2. In Colab, click the **key icon** in the left sidebar (Secrets).
3. Add a secret named `GITHUB_TOKEN`, paste the token as the value, and switch
   **Notebook access** on.

Never paste the token into a cell — it would be saved into the notebook and
pushed to GitHub. The cell below reads it from Secrets, uses it only for the
clone, and then rewrites the remote so the token is not stored in `.git/config`.

Making the repository public instead also works, and needs no token.

In [ ]:
import os
import subprocess

REPO_PATH = 'conradpower279/Thalamus'
REPO_URL = f'https://github.com/{REPO_PATH}.git'
REPO_DIR = '/content/Thalamus'

# Read the token from Colab Secrets. Missing secret, or notebook access not
# granted, both just mean "no token" - handled below.
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    print(f'No GITHUB_TOKEN from Colab Secrets ({type(exc).__name__}).')

HELP = f"""
Could not reach {REPO_PATH}, which is a private repository.

Either:
  1. Add a GitHub token to Colab Secrets (key icon in the left sidebar):
       Name:  GITHUB_TOKEN
       Value: a token from https://github.com/settings/tokens
              fine-grained -> Contents: Read-only on this repo
              classic      -> 'repo' scope
     Turn ON "Notebook access", then re-run this cell.

  2. Or make the repository public on GitHub, then re-run this cell.
"""


def run_git(args, cwd=None):
    """Run git, returning (ok, output) with any token masked out."""
    result = subprocess.run(['git'] + args, cwd=cwd, capture_output=True, text=True)
    output = (result.stdout + result.stderr)
    if token:
        output = output.replace(token, '***')
    return result.returncode == 0, output


# The token goes in the URL for the clone only. Afterwards the remote is reset to
# the clean URL, so the token never lands in .git/config.
auth_url = f'https://{token}@github.com/{REPO_PATH}.git' if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned, pulling latest')
    ok, output = run_git(['pull', auth_url, 'main'], cwd=REPO_DIR)
    print(output)
    if not ok:
        raise SystemExit(HELP)
elif os.path.exists(REPO_DIR) and os.listdir(REPO_DIR):
    raise SystemExit(
        f'{REPO_DIR} exists and is not a git checkout. Remove it and re-run:\n'
        '    !rm -rf /content/Thalamus'
    )
else:
    print(f'Cloning {REPO_PATH}' + (' (using GITHUB_TOKEN)' if token else ' anonymously'))
    ok, output = run_git(['clone', auth_url, REPO_DIR])
    print(output)
    if not ok:
        raise SystemExit(HELP)
    run_git(['remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR)

os.chdir(REPO_DIR)

# Confirm the code is really here before any later cell depends on it.
if not os.path.isfile('scripts/fish_classifier_train.py'):
    raise SystemExit(HELP)
print('\nRepo ready at', os.getcwd())

In [ ]:
# timm/albumentations/onnx pinned to match requirements.txt; torch comes from Colab.
!pip install -q timm==1.0.11 albumentations==1.4.21 onnx==1.17.0 PyYAML==6.0.2

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Get raw images

**Choose one of 3A or 3B.** Both produce the same thing: a folder of raw images,
one subfolder per species, at `/content/raw_fish`. Step 4 onward does not care
which route you took.

### 3A. Build the Florida fish dataset from iNaturalist

Filters the [iNaturalist Open Dataset](https://github.com/inaturalist/inaturalist-open-data)
down to ray-finned fish (Actinopterygii) observed in Florida waters, then
downloads the matching research-grade photos.

This pulls several GB of gzipped metadata and can take a while. **Skip to 3B if
you already have a dataset in Drive.**

In [ ]:
# Pull the iNaturalist metadata (~several GB of gzipped CSVs).
!pip install -q duckdb pandas boto3 awscli

import os

INAT_DATA_DIR = '/content/inat_data'
os.makedirs(INAT_DATA_DIR, exist_ok=True)

for name in ['taxa', 'observations', 'photos']:
    dest = f'{INAT_DATA_DIR}/{name}.csv.gz'
    if not os.path.exists(dest):
        !aws s3 cp s3://inaturalist-open-data/{name}.csv.gz {dest} --no-sign-request
    else:
        print(f'{dest} already present, skipping')

In [ ]:
# Filter the metadata to Florida ray-finned fish, then download the photos.
import os
import duckdb
from concurrent.futures import ThreadPoolExecutor

# ---- CONFIG ----
# Rough bounding box for Florida coastal / offshore waters.
FLORIDA_BBOX = {'min_lat': 24.3, 'max_lat': 31.1, 'min_lon': -87.7, 'max_lon': -79.7}

# Per-species caps. Drop species with too few photos to learn from, and cap the
# common ones so the dataset stays balanced (and the download stays reasonable).
MIN_PER_SPECIES = 50    # skip species with fewer than this many matching photos
MAX_PER_SPECIES = 300   # download at most this many photos per species

# Raw images live outside the repo's data/ directory, which holds the splits.
OUTPUT_DIR = '/content/raw_fish'
os.makedirs(OUTPUT_DIR, exist_ok=True)

con = duckdb.connect(database=':memory:')

# DuckDB reads gzipped CSVs directly, no decompression needed.
for view in ['taxa', 'observations', 'photos']:
    con.execute(f"""
        CREATE VIEW {view} AS
        SELECT * FROM read_csv_auto('{INAT_DATA_DIR}/{view}.csv.gz', delim='\t', header=True)
    """)

# `taxa.ancestry` is a slash-delimited string of taxon IDs (e.g. '48460/1/2/355675'),
# NOT names. So look up the taxon_id for the ray-finned fish class first, then
# match descendants whose ancestry contains that id.
fish_taxon_id = con.execute(
    "SELECT taxon_id FROM taxa WHERE name = 'Actinopterygii'"
).fetchone()
if fish_taxon_id is None:
    raise SystemExit("Could not find the 'Actinopterygii' taxon — check taxa.csv.gz")
fish_taxon_id = fish_taxon_id[0]
print(f'Actinopterygii taxon_id = {fish_taxon_id}')

# Fish class (Actinopterygii, by id) + Florida bounding box + research-grade only.
# Then apply per-species caps with window functions:
#   - species_total  : how many photos the species has (for the MIN filter)
#   - rn             : per-species row number (for the MAX cap)
#
# rank = 'species' keeps genus- and family-level identifications out. Without it
# you get folders like 'Caranx' sitting alongside 'Caranx_crysos', which are not
# mutually exclusive classes and confuse both training and evaluation.
query = f"""
    WITH matched AS (
        SELECT
            o.observation_uuid,
            t.name AS species_name,
            o.latitude,
            o.longitude,
            p.photo_id,
            p.extension
        FROM observations o
        JOIN taxa t ON o.taxon_id = t.taxon_id
        JOIN photos p ON o.observation_uuid = p.observation_uuid
        WHERE ('/' || t.ancestry || '/') LIKE '%/{fish_taxon_id}/%'
          AND t.rank = 'species'
          AND o.latitude BETWEEN {FLORIDA_BBOX['min_lat']} AND {FLORIDA_BBOX['max_lat']}
          AND o.longitude BETWEEN {FLORIDA_BBOX['min_lon']} AND {FLORIDA_BBOX['max_lon']}
          AND o.quality_grade = 'research'
    ),
    ranked AS (
        SELECT
            *,
            COUNT(*)     OVER (PARTITION BY species_name) AS species_total,
            ROW_NUMBER() OVER (PARTITION BY species_name ORDER BY photo_id) AS rn
        FROM matched
    )
    SELECT observation_uuid, species_name, latitude, longitude, photo_id, extension
    FROM ranked
    WHERE species_total >= {MIN_PER_SPECIES}
      AND rn <= {MAX_PER_SPECIES}
"""

results = con.execute(query).fetchdf()
print(
    f"Selected {len(results)} photos across {results['species_name'].nunique()} species "
    f"(min {MIN_PER_SPECIES}/species, capped at {MAX_PER_SPECIES})"
)
results.to_csv('/content/florida_fish_photo_list.csv', index=False)
print(results['species_name'].value_counts().head(30))

In [ ]:
# Download the photos, in parallel, unsigned.
import boto3
from botocore import UNSIGNED
from botocore.config import Config

MAX_WORKERS = 16
BUCKET = 'inaturalist-open-data'

# Size the HTTP connection pool to the worker count, otherwise boto3's default
# pool of 10 is too small for our threads and logs "Connection pool is full,
# discarding connection" while it churns connections instead of reusing them.
s3 = boto3.client('s3', config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=MAX_WORKERS,
))


def download_photo(row, size='medium'):
    species_dir = os.path.join(OUTPUT_DIR, row['species_name'].replace(' ', '_'))
    os.makedirs(species_dir, exist_ok=True)

    # The observation_uuid goes in the FILENAME on purpose. One iNaturalist
    # observation often has several photos of the same individual fish, minutes
    # apart. If those land on both sides of the train/val split, validation
    # accuracy measures memorisation and can be inflated by tens of points.
    # Step 5 groups on this prefix so a whole observation stays in one split.
    dest = os.path.join(
        species_dir, f"{row['observation_uuid']}_{row['photo_id']}.{row['extension']}"
    )
    if os.path.exists(dest):
        return
    key = f"photos/{row['photo_id']}/{size}.{row['extension']}"
    try:
        s3.download_file(BUCKET, key, dest)
    except Exception as exc:
        print(f'  skip {row["photo_id"]}: {exc}')


rows = results.to_dict('records')
print(f'Downloading {len(rows)} photos to {OUTPUT_DIR} ...')
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for i, _ in enumerate(pool.map(download_photo, rows), 1):
        if i % 5000 == 0:
            print(f'  {i}/{len(rows)} processed')

total = sum(len(files) for _, _, files in os.walk(OUTPUT_DIR))
print(f'Done. {total} images under {OUTPUT_DIR}, one folder per species.')

### 3B. Or load a prepared dataset from Drive

Use this instead of 3A if you already have a dataset archive. It should unpack to
one folder per species.

Colab's local disk is wiped when the runtime recycles, so keep the dataset in
Drive. Copying a zip and unpacking locally is far faster than reading thousands
of small files off a mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust to wherever your dataset archive lives in Drive.
ARCHIVE = '/content/drive/MyDrive/thalamus/fish_dataset.zip'

!mkdir -p /content/raw_fish
!unzip -q -o {ARCHIVE} -d /content/raw_fish
!ls /content/raw_fish

## 4. Prepare the images

Source images can be any size, any aspect ratio — they are resized as they load,
so nothing needs editing by hand. This step handles what resizing-on-load cannot:
files that are corrupt, formats the loader does not read, and multi-megapixel
photos that would be decoded down to 224px on every epoch of every run.

First audit. This changes nothing.

In [ ]:
!python scripts/prepare_images.py --source /content/raw_fish

Then write a cleaned, size-capped copy. The originals are left untouched.

iNaturalist photos in particular arrive in mixed formats and dimensions, and at
this download volume some files are always truncated — this is where they get
caught rather than interrupting an epoch two hours in.

In [ ]:
!python scripts/prepare_images.py \
    --source /content/raw_fish \
    --output /content/raw_fish_prepared \
    --max-size 640

## 5. Split into train / val / test

Turns one-folder-per-species into the `train/val/test` layout training expects.

`--group-regex` is doing important work here. The filenames from step 3A start
with the iNaturalist observation UUID, and one observation often holds several
photos of the same individual fish. Grouping on that prefix keeps a whole
observation inside a single split. Without it, near-duplicate photos land on both
sides and validation accuracy stops meaning anything.

If you came via 3B and your filenames have no group prefix, drop the
`--group-regex` line — but do read the leakage note in `data/README.md` first.

In [ ]:
!python scripts/split_data.py \
    --source /content/raw_fish_prepared \
    --output /content/Thalamus/data \
    --group-regex '^([0-9a-f-]{36})_'

## 6. Verify the layout

**Stop here if this cell fails.** It checks exactly what the training script
needs, and a clear failure now beats a confusing one after the model has loaded.

In [ ]:
from pathlib import Path

DATA = Path('/content/Thalamus/data')
problems = []

for split in ['train', 'val']:
    if not (DATA / split).is_dir():
        problems.append(f'missing directory: {DATA / split}')

if not problems:
    def species_counts(split):
        return {
            d.name: len([p for p in d.iterdir()
                         if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}])
            for d in sorted((DATA / split).iterdir()) if d.is_dir()
        }

    train_counts, val_counts = species_counts('train'), species_counts('val')

    if not train_counts:
        problems.append('no species directories under data/train')
    if set(train_counts) != set(val_counts):
        only_train = sorted(set(train_counts) - set(val_counts))
        only_val = sorted(set(val_counts) - set(train_counts))
        problems.append(f'train/val species differ — only in train: {only_train}, '
                        f'only in val: {only_val}')

    empty_val = [name for name, n in val_counts.items() if n == 0]
    if empty_val:
        problems.append(f'{len(empty_val)} species have no validation images: '
                        f'{empty_val[:10]}')

    print(f'{len(train_counts)} species | '
          f'{sum(train_counts.values())} train, {sum(val_counts.values())} val\n')
    for name in sorted(train_counts, key=lambda k: -train_counts[k])[:20]:
        print(f'  {name:<40} train={train_counts[name]:>6}  val={val_counts.get(name, 0):>5}')

    if train_counts:
        biggest, smallest = max(train_counts.values()), min(train_counts.values())
        if smallest:
            print(f'\n  imbalance ratio {biggest / smallest:.0f}:1 '
                  '(unweighted loss favours the common species)')

if problems:
    print('\nNOT READY TO TRAIN:')
    for problem in problems:
        print(f'  - {problem}')
    raise SystemExit('Fix the above, then re-run step 5.')
print('\nLayout OK — ready to train.')

## 7. Configure

Defaults come from `configs/default.yaml` (MobileNetV3). `configs/mobilenetv4.yaml`
selects MobileNetV4-Conv-Small instead and writes to separate checkpoint files, so
the two can be trained and compared without overwriting each other.

In [ ]:
import yaml

CONFIG_PATH = 'configs/default.yaml'      # or 'configs/mobilenetv4.yaml'

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

for key, value in config.items():
    print(f'{key:>20}: {value}')

## 8. Train

Stage 1 trains the classifier head on the frozen backbone; stage 2 unfreezes
everything and fine-tunes on a cosine schedule. Checkpoints, `classes.json` and
the ONNX export land in `models/`.

Colab disconnects idle sessions — keep the tab visible for long runs.

In [ ]:
!python scripts/fish_classifier_train.py \
    --config {CONFIG_PATH} \
    --data-dir /content/Thalamus/data

## 9. Back up the checkpoints

Do this before the runtime recycles, or the run is lost. `classes.json` matters as
much as the weights: the model emits bare indices and cannot name a species
without it.

In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/thalamus/models
!cp -v models/*.pth models/*.onnx models/classes.json /content/drive/MyDrive/thalamus/models/

## 10. Export to TFLite for the Flutter app

The conversion toolchain is large and pins versions that conflict with the
training environment, so install it last. Expect to restart the runtime
afterwards — if you do, re-run step 2 before coming back here.

For an int8 model pass `--calib-dir /content/Thalamus/data/val`; calibration needs
real images.

In [ ]:
!pip install -q -r requirements-export.txt

In [ ]:
!python export/export_tflite.py --quantization float16
!ls -lh export/output/

## 11. Download the app artifacts

Always keep the `.tflite` and the `classes.json` it was built with together — the
class list comes from sorted species folder names, so adding a species later
shifts every index after it.

In [ ]:
from google.colab import files
from pathlib import Path

for path in sorted(Path('export/output').glob('*.tflite')):
    files.download(str(path))

files.download('export/output/classes.json')